# 03 · The Random Walk & Geometric Brownian Motion
**Goal:** understand the standard 'honest' model of a price — a **random walk** — and its continuous form **Geometric Brownian Motion (GBM)**, which is how every project simulates prices when it doesn't want real data.

> Maps to: projects **07, 29, 30** (all simulate prices). KB §1.5.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2)

## 1. A random walk
Each step the value moves up or down by a random amount, *independent of the past*. You genuinely cannot predict the next step — which is roughly how efficient markets behave.

In [ ]:
n = 252
for _ in range(5):
    steps = rng.choice([-1, 1], size=n)
    plt.plot(np.cumsum(steps))
plt.title('5 random walks (no two alike, none predictable)'); plt.show()

## 2. Geometric Brownian Motion
GBM is the standard price model where *returns* are normal. The price is:

$$P_t = P_0\,\exp\!\big((\mu - \tfrac12\sigma^2)t + \sigma W_t\big)$$

- `mu` = **drift** (long-run growth)
- `sigma` = **volatility** (size of the noise)
- the `-0.5*sigma**2` is a technical correction so the average comes out right.

In [ ]:
def gbm(s0=100, mu=0.08, sigma=0.2, n=252, seed=None):
    r = np.random.default_rng(seed)
    dt = 1/252
    shocks = (mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*r.normal(size=n)
    return s0 * np.exp(np.cumsum(shocks))

plt.figure(figsize=(9,3))
for s in range(6):
    plt.plot(gbm(sigma=0.2, seed=s))
plt.title('6 simulated GBM price paths (mu=8%, sigma=20%)'); plt.ylabel('price'); plt.show()

## 3. Monte Carlo: run it many times
**Monte Carlo** just means 'simulate the random scenario thousands of times and look at the *distribution* of outcomes'. Let's see where the price ends up after a year across 5,000 simulations.

In [ ]:
ends = np.array([gbm(sigma=0.2, seed=s)[-1] for s in range(5000)])
plt.figure(figsize=(8,3))
plt.hist(ends, bins=50); plt.axvline(100, color='k', ls='--', label='start = 100')
plt.title('Year-end price across 5,000 simulations'); plt.legend(); plt.show()
print(f'median end price: {np.median(ends):.1f}')
print(f'5th percentile (a bad year): {np.percentile(ends,5):.1f}')
print(f'95th percentile (a good year): {np.percentile(ends,95):.1f}')

## 4. More examples: probability of a loss, and the effect of volatility
Monte Carlo lets us answer concrete questions by counting simulated outcomes.

In [ ]:
def end_prices(sigma, k=5000):
    return np.array([gbm(sigma=sigma, seed=s)[-1] for s in range(k)])

for sigma in [0.10, 0.20, 0.40]:
    ends = end_prices(sigma)
    p_loss = np.mean(ends < 100)
    print(f'sigma={sigma:.0%}:  P(end below start) = {p_loss:.1%}   '
          f'5th pct = {np.percentile(ends,5):6.1f}   95th pct = {np.percentile(ends,95):6.1f}')

Higher volatility widens the cone of outcomes — both the bad 5th percentile and the good 95th get more extreme, and the chance of ending below where you started rises. This is the engine behind Monte Carlo VaR (the risk project's concepts) and execution simulation (the execution project's concepts).

### 🧪 Try it yourself
1. Increase the drift `mu` to `0.20` — the loss probability falls (growth fights the noise).
2. Estimate the probability the price ever *doubles* during the year: `np.mean([gbm(seed=s).max() > 200 for s in range(2000)])`.
3. Raise the number of simulations from 5000 to 50000 — the estimates steady (more samples = less Monte Carlo noise).

**You should see:** individual paths look wild and unpredictable, but the *distribution* of outcomes is stable and informative — that's the whole idea behind Monte Carlo risk and execution simulation.

### In the projects
- GBM-style synthetic prices → project **07** `src/data.py:make_synthetic_prices`.
- Monte Carlo over price paths → project **29** `src/simulator.py:monte_carlo` and project **07** `src/var.py:monte_carlo_var`.